# LAB 2:  AutoML Tables による出生体重（Babyweight）予測モデルのトレーニング

**学習目標**

1. Vertex AI のセットアップ
1. BigQuery からデータセットを作成してインポート
1. データセットの分析
1. AutoML Tables モデルのトレーニング
1. 評価指標の確認
1. モデルのデプロイ
1. バッチ予測の実行
1. オンライン予測の実行


## はじめに 
このノートブックでは、AutoML Tables を使用して、赤ちゃんが生まれる前の体重を予測するモデルをトレーニングします。Vertex AI UI を使用して BigQuery からトレーニング データセットを作成し、AutoML Tables モデルでトレーニング、評価、予測を行います。

このラボでは、AutoML Tables のセットアップ、BigQuery からの AutoML Tables データセットの作成とインポート、AutoML Tables データセットの分析、AutoML Tables モデルのトレーニング、トレーニング済みモデルの評価指標の確認、トレーニング済みモデルのデプロイを行い、最後にトレーニング済みモデルを使用してバッチ予測とオンライン予測の両方を実行します。

各学習目標は、この演習用ラボ ノートブックで完了する一連のステップに対応しています。

## テーブルの存在確認

次のセルを実行して、データセットとデータ テーブルが以前に作成されていることを確認します。作成されていない場合は、ラボ [1b_prepare_data_babyweight](../solutions/1b_prepare_data_babyweight.ipynb) に戻って作成してください。

In [ ]:
%load_ext google.cloud.bigquery

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリであり、テーブルが存在するかどうかを確認できます。
SELECT * FROM babyweight.babyweight_augmented_data
LIMIT 0

## ラボ課題 1: Vertex AI のセットアップ

### ステップ 1: Vertex AI を開く
GCP コンソールに移動し、左上隅にあるコンソール メニューを開きます。下部にスクロールして「人工知能（Artificial Intelligence）」セクションに移動します。`Vertex AI` をクリックして開きます。

<img src="../assets/vertex/01_vertexai_hamburger_dropdown.png" width='70%'>

### ステップ 2: API の有効化
Vertex AI API をまだ有効にしていない場合は、以下の画面が表示されます。`ENABLE VERTEX AI API` ボタンをクリックしてください。

<img src="../assets/vertex/02_vertexai_enable_api.png" width='70%'>

## ラボ課題 2: BigQuery からデータセットを作成してインポート

### ステップ 3: データセット
左側のメニューで `Datasets` をクリックします。

<img src="../assets/vertex/03_datasets.png" width='70%'>

Vertex AI の Datasets ページが表示されます。ここには、インポートされたすべてのデータセットが表示されます。出生体重データセットを追加したいので、`+ CREATE` ボタンをクリックします。

<img src="../assets/vertex/04_create_dataset.png" width='70%'>

### ステップ 4: 新しいデータセットを作成
新しいデータセットに一意の名前を付ける必要があります。ここでは `babyweight_automl` と名付けましたが、好きな名前を付けて構いません。回帰モデルを構築するため、`TABULAR` と `Regression/classification` を選択します。一意の名前を選択したら、`CREATE` ボタンをクリックします。

<img src="../assets/vertex/05_dataset_name.png" width='70%'>

### ステップ 5: データのインポート
データセットを作成したので、Vertex AI と AutoML Tables がトレーニングに使用できるようにデータをインポートしましょう。データは現在すでに BigQuery にあるため、ラジオボタン `Select a table or view from BigQuery` を選択します。これにより、BigQuery テーブルを検索するためのテキスト ボックスが表示されます。このフィールドには `project.dataset.table_name` の形式が必要ですが、下に示すように table_name を入力するだけでテーブルを検索できます。入力が完了したら、`CONTINUE` ボタンをクリックします。

<img src="../assets/vertex/06_select_bigquery_table.png" width='70%'>

<img src="../assets/vertex/07_selected_bigquery_table.png" width='70%'>

## ラボ課題 3: データセットの分析

### ステップ 6: データセットの分析
`GENERATE STATISTICS` をクリックすると、ここでいくつかの基本的な統計情報を分析できます。6 つの特徴量があり、そのうち 4 つが数値型、2 つがカテゴリ型であることがわかります。また、すべての列に欠損値がないことも確認できます。これは素晴らしいことです！さらに、個別値（個別ユニーク値）の数も確認でき、予想と比較することができます。分析に満足したら、**TRAIN NEW MODEL** をクリックします。（データ品質に十分な自信がある場合は、統計を実行せずに、または統計の生成が完了する前にモデルのトレーニングを開始することもできます。）

<img src="../assets/vertex/08_generate_statistics.png" width='70%'>

## ラボ課題 4: AutoML Tables モデルのトレーニング

### ステップ 7: モデルタイプの設定とターゲット列の選択
`Training method` セクションで、目的として `Regression` を選択し、モデルタイプとして `AutoML` を選択して、`Continue` をクリックします。

<img src="../assets/vertex/09_regression.png" width='70%'>

`Model details` セクションで、Target Column として `weight_pounds` を選択し、`Continue` をクリックします。

<img src="../assets/vertex/10_target_column.png" width='70%'>

### ステップ 8: 入力特徴量の選択
6 つの列をインポートし、そのうちの 1 つである `weight_pounds` をターゲット（ラベル）列として確保しました。残り 5 つの列があります。`Training options` ドロップダウンには、すべての列のリストが表示されます。4 つの特徴量として `is_male`、`mother_age`、`plurality`、`gestation_weeks` を使用します。`hashmonth` は、[2_prepare_babyweight](../solutions/2_prepare_babyweight.ipynb) ラボで再現性のあるデータ分割を行ったときの残りであり、ここでは不要です。右側のアイコンをクリックして除外してください。

<img src="../assets/vertex/11_feature_selection.png" width='70%'>

### ステップ 9: トレーニングのセットアップ
モデルをトレーニングする準備がほぼ整いました。ここに至るまでに多くのステップが必要でしたが、それらは主にデータのインポートと、データに問題がないことを確認するためのものでした。周知の通り、ML においてデータは極めて重要であり、データが期待どおりでなければ、モデルも期待どおりに機能しません（Garbage in, garbage out: ゴミを入れたらゴミしか出ない）。モデルのトレーニングに費やす最大ノード時間である `Training budget` を設定する必要があります。幸いなことに、それより前に改善が止まった場合は（`Enable early stopping` トグルがオンの場合）、トレーニングは停止し、実際に使用したノード時間に対してのみ課金されます。このデータセットでは、わずか 1 ノード時間の予算で十分な結果が得られました。設定が完了したら、`START TRAINING` ボタンをクリックします。

<img src="../assets/vertex/12_budget.png" width='70%'>

### ステップ 10: モデルのトレーニングを待つ
モデルのトレーニングが始まりました！完了すれば素晴らしいモデルができるはずです！そして今は待ちます。データセットのサイズ、トレーニング予算、その他の要因によって、数時間から丸 1 日以上かかる場合があるため、このステップは辛抱強く待つことになります。待っている間の良い過ごし方としては、このシリーズの次のラボを進めておき、モデルのトレーニングが完了したらこのラボに戻ってくることです。

<img src="../assets/vertex/13_automl_tables_training.png" width='70%'>

## ラボ課題 5: 評価指標の確認

### ステップ 11: モデルの評価
トレーニングが完了しました！トレーニング ジョブが完了したことを知らせるメールが届いたら、上記で見たように Dataset ページをもう一度開いて（完了している場合はモデル名をクリック）、モデルのページに移動するか、左側のサイドバーで `Vertex AI`、次に `Models` を選択して、最近作成されたモデル（その `Type` は `Tabular` になります）を探します。

これで `EVALUATE` タブを確認し、どれくらいうまくいったかを確認できます。ターゲットが `weight_pounds` であったこと、トレーニングが `RMSE` 向けに最適化されたこと、そして MAE や RMSE などの多くの評価指標が表示されます。筆者のトレーニングの実行結果は良好で、わずか 1 時間のトレーニングで RMSE は 1.027 でした！AutoML の驚くべき威力がよく分かります。下には特徴量の重要度の棒グラフが表示されています。`gestation_weeks` が圧倒的に最も重要ですが、これは妊娠期間が長いほど赤ちゃんが成長する時間が長く、したがって赤ちゃんの体重が重くなるのが普通であるため、理にかなっています。

<img src="../assets/vertex/14_automl_tables_evaluate.png" width='70%'>

## ラボ課題 6: バッチ予測の実行

### ステップ 12: バッチ予測ジョブの作成
新しく作成した AutoML Tables モデルのトレーニングと評価の成果に満足できたら、次は ML の真骨頂である高精度な予測の実行に進みましょう！モデルの準備ができたので、オンライン予測用にモデルをデプロイしなくても、バッチ予測を作成できます。簡単にするために、ここではトレーニングと評価に使用した BigQuery テーブルに対して予測を行うことにします。これを行うには、`BATCH PREDICTIONS` タブを選択し、`CREATE BATCH PREDICTION` をクリックして、任意の `Batch prediction name` を指定します。ラジオボタン `BigQuery table` を選択し、`BigQuery Project Id`、`BigQuery Dataset Id`、`BigQuery Table or View Id` を入力します。Google Cloud Storage の CSV を使用することもできます。次に、結果の出力先を選択する必要があります。Output format として `BigQuery` を選択し、`Google Cloud project ID` を入力します。結果を Google Cloud Storage に書き込むこともできます。すべて設定したら、`CREATE` をクリックします。これにより、トレーニング済みの AutoML 表形式モデルと上記で選択した場所のデータを使用してバッチ予測ジョブが送信されます。

<img src="../assets/vertex/15_automl_tables_batch_predict.png" width='70%'>

### ステップ 13: バッチ予測ジョブの完了
少し待つと、バッチ予測が完了するはずです。筆者のデータセットでは 30 分もかかりませんでした。先ほどの画面で完了したジョブを確認するか、ハンバーガー メニューから `Vertex AI` と `Batch predictions` に移動します。バッチ予測の名前をクリックします。データ入力、結果の保存場所、作成日時、処理にかかった時間が表示されます。`Export location` のリンクをクリックして [BigQuery Console UI](https://console.cloud.google.com/bigquery) に移動し、確認してみましょう。

<img src="../assets/vertex/18_automl_tables_batch_predict_results.png" width='70%'>

### ステップ 14: バッチ予測データセット
左端のプロジェクト リストに、作業中のプロジェクトが表示されます。矢印をクリックして、プロジェクト内のすべての BigQuery データセットのドロップダウン リストを展開します。前のステップの `Results directory` に表示されていたものと同じ新しいデータセットが表示されます。そのデータセットのドロップダウン リストを展開すると、作成された 2 つの BigQuery テーブル `predictions` と `errors` が表示されます。

<img src="../assets/vertex/19_automl_tables_batch_predict_dataset.png" width='70%'>

### ステップ 15: バッチ予測の予測結果
まず `predictions` テーブルを見てみましょう。`PREVIEW` タブをクリックして内容を表示します。基本的には、バッチ予測ジョブへの入力データを受け取り、それに 3 つの新しい列を追加しています。`hashmonth` のように、モデルの特徴量として使用しなかった列もここに含まれていることに注目してください。`start` と `end` の 2 つの `prediction_inteval` 列が表示されているはずです。最後の列は予測 `value` であり、これは行内の対応する特徴量を使用してトレーニング済みの AutoML Tables モデルによって計算された、予測された `weight_pounds` です。

<img src="../assets/vertex/20_automl_tables_batch_predict_prediction_table.png" width='70%'>

### ステップ 16: バッチ予測のエラー
エラーが発生していないか `errors` テーブルを確認することもできます。筆者がバッチ予測ジョブを実行したときは、幸いにもエラーはありませんでしたが、エラーが発生した場合には間違いなくここを確認するべきです。筆者の `errors` テーブルは空だったため、以下にスキーマを示します。ここでも基本的には、バッチ予測ジョブへの入力データを受け取り、それに 3 つの新しい列を追加しています。レコードが保存されるほか、エラー `code` と `error` メッセージも記録されます。これらは予期しない動作をデバッグする際に役立ちます。

<img src="../assets/vertex/21_automl_tables_batch_predict_errors_table_schema.png" width='70%'>

## ラボ課題 7: モデルのデプロイ

### ステップ 15: オンライン予測用モデルのデプロイ
上記のバッチ ジョブよりも高速に予測を取得したい場合もあるでしょう。そのためには、トレーニング済みモデルをデプロイする必要があります。`DEPLOY & TEST` タブに移動し、`DEPLOY TO ENDPOINT` をクリックします。

<img src="../assets/vertex/16_automl_tables_deploy.png" width='70%'>

### ステップ 17: デプロイ設定

Endpoint に名前を付けます。残りの設定はデフォルトのままにして、`DEPLOY` をクリックします。

<img src="../assets/vertex/17_deploy_options.png" width='70%'>

## ラボ課題 8: オンライン予測の実行

### ステップ 18: オンライン予測のセットアップ
デプロイが完了すると、`DEPLOY TO ENDPOINT` ボタンの下にエンドポイントが表示され、さらに下にあるモデルの特徴量テーブルを含む `Test your model` 領域がアクティブになります。各特徴量には、列名、データ型、必須かどうか、事前に入力された値が表示されます。これらの値はそのままにしておくことも、別の値を入力することもできます。カテゴリ特徴量の場合は、必ず有効な値を使用してください。そうしないと OOV（語彙外）のスピルオーバーになり、トレーニングの成果を最大限に活用できなくなります。値の設定が完了したら、`PREDICT` ボタンをクリックします。

<img src="../assets/vertex/22_automl_tables_before_online_predict.png" width='70%'>

### ステップ 19: オンライン予測の結果
しばらくすると、画面にオンライン予測結果が表示されます。`Prediction result` と `95% prediction interval` が返されます。各特徴量に他の値を試してみて、どのような予測結果になるか確認してみてください！

<img src="../assets/vertex/23_automl_tables_after_online_predict.png" width='70%'>

## ラボのまとめ: 
このラボでは、AutoML Tables のセットアップ、BigQuery からの AutoML Tables データセットの作成とインポート、AutoML Tables データセットの分析、AutoML Tables モデルのトレーニング、トレーニング済みモデルの評価指標の確認、トレーニング済みモデルのデプロイを行い、トレーニング済みモデルを使用してバッチ予測とオンライン予測の両方を実行しました。

Copyright 2021 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.